# 🧠 Semantic State Networks for LLM Generation Quality Prediction

## A Network Data Science Approach to Detecting and Preventing Semantic Degradation

---

### Research Question

> **Can we mathematically predict when an LLM's generation quality will degrade using network-theoretic metrics?**

We model LLM text generation as a **Markov process on a semantic state network**, where real-time monitoring of network properties enables early detection of quality degradation and dynamic intervention.

### Key Innovation

Rather than detecting hallucinations (factually incorrect but fluent text), we focus on **semantic degradation patterns** - the structural breakdown of coherent text generation that often precedes or accompanies hallucination. This includes:
- Repetitive loops
- Topic drift  
- Incoherent transitions
- Loss of semantic structure

### Network Science Framework

| Network Concept | Application | Mathematical Formulation |
|----------------|-------------|-------------------------|
| **Markov Chains** | Model text as state transitions | $P(s_{t+1} \| s_t)$ |
| **Transition Matrices** | Quantify semantic flow | $T_{ij} = P(j \| i)$ |
| **Network Entropy** | Measure generation chaos | $H = -\sum p_{ij} \log p_{ij}$ |
| **Spectral Gap** | Analyze convergence speed | $\gamma = 1 - \|\lambda_2\|$ |
| **Stationary Distribution** | Long-term state occupancy | $\pi T = \pi$ |

### Models Under Study
- **GPT-2** (124M parameters) - Baseline smaller model
- **Gemma-2-2B** (2.7B parameters) - Modern instruction-tuned model

### Hypotheses

1. **H1**: Network entropy correlates positively with generation degradation
2. **H2**: Spectral gap predicts generation stability  
3. **H3**: Dynamic temperature control based on entropy can reduce degradation
4. **H4**: These patterns are model-agnostic (hold across GPT-2 and Gemma-2)

## 1. Environment Setup

In [ ]:
import torch
import pandas as pd
import numpy as np
import time
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import math

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['font.size'] = 11

COLORS = {
    'coherent': '#2ecc71', 'neutral': '#3498db', 'degraded': '#e74c3c',
    'autopilot': '#9b59b6', 'gpt2': '#e67e22', 'gemma': '#1abc9c'
}
STATE_NAMES = ['Coherent', 'Neutral', 'Degraded']
STATE_COLORS = [COLORS['coherent'], COLORS['neutral'], COLORS['degraded']]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {DEVICE.upper()}")

In [ ]:
def load_models():
    tok_gpt2 = AutoTokenizer.from_pretrained("gpt2")
    mod_gpt2 = AutoModelForCausalLM.from_pretrained("gpt2").to(DEVICE)

    try:
        dtype = torch.bfloat16 if DEVICE == "cuda" else torch.float32
        tok_gemma = AutoTokenizer.from_pretrained("google/gemma-2-2b")
        mod_gemma = AutoModelForCausalLM.from_pretrained("google/gemma-2-2b", torch_dtype=dtype).to(DEVICE)
    except Exception as e:
        print(e)
        return None, None, None

    embedder = SentenceTransformer('all-MiniLM-L6-v2')

    return (tok_gpt2, mod_gpt2), (tok_gemma, mod_gemma), embedder

(tok_gpt2, mod_gpt2), (tok_gemma, mod_gemma), embedder = load_models()

## 2. Semantic State Calibration

In [ ]:
def get_calibration_data():
    coherent = [
        "The standard model of particle physics describes three of the four fundamental forces.",
        "To solve the differential equation, we apply the Laplace transform to both sides.",
        "Photosynthesis converts light energy into chemical energy stored in glucose molecules.",
        "The neural network architecture uses multi-head attention followed by feedforward layers.",
        "Climate change mitigation requires renewable energy adoption and carbon capture technology.",
        "Quantum entanglement enables instantaneous correlation between distant particles.",
        "Reinforcement learning agents maximize cumulative reward through trial and error.",
        "Transformer models use self-attention to capture long-range dependencies in sequences."
    ] * 25

    neutral = [
        "However, this is not the only case to consider.",
        "Let us now examine the following example.",
        "The result is shown in the figure below.",
        "Moving on to the next important point.",
        "This depends on the specific context.",
        "The answer is essentially yes.",
        "There are several factors to consider.",
        "As mentioned earlier in the text."
    ] * 25

    degraded = [
        "The the the the the the the the.",
        "Blue sky pizza algorithm running jump syntax.",
        "Cat dog mouse house mouse cat dog dog.",
        "Training loss NaN NaN NaN NaN.",
        "A A A A B B B B C C C C.",
        "Generate produce make build create generate.",
        "Error error error stack overflow memory.",
        "Random word salad mixture nonsense output."
    ] * 25

    labels = [0]*len(coherent) + [1]*len(neutral) + [2]*len(degraded)
    return coherent + neutral + degraded, labels

calibration_texts, true_labels = get_calibration_data()
print(f"Samples: {len(calibration_texts)}")

In [ ]:
print("Computing embeddings...")
calibration_embeddings = embedder.encode(calibration_texts, show_progress_bar=True)

print("Fitting K-Means...")
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(calibration_embeddings)

silhouette = silhouette_score(calibration_embeddings, cluster_labels)
print(f"Silhouette Score: {silhouette:.3f}")

In [ ]:
# Semantic State Space Visualization
pca = PCA(n_components=2)
coords = pca.fit_transform(calibration_embeddings)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for i, (name, color) in enumerate(zip(STATE_NAMES, STATE_COLORS)):
    mask = cluster_labels == i
    axes[0].scatter(coords[mask, 0], coords[mask, 1], c=color, label=name, alpha=0.6, s=50)

centroids_2d = pca.transform(kmeans.cluster_centers_)
axes[0].scatter(centroids_2d[:, 0], centroids_2d[:, 1], c='black', marker='X', s=200, zorder=5)
axes[0].set_title('Learned Clusters', fontweight='bold')
axes[0].legend()
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')

for i, (name, color) in enumerate(zip(STATE_NAMES, STATE_COLORS)):
    mask = np.array(true_labels) == i
    axes[1].scatter(coords[mask, 0], coords[mask, 1], c=color, label=name, alpha=0.6, s=50)
axes[1].set_title('Ground Truth', fontweight='bold')
axes[1].legend()
axes[1].set_xlabel(f'PC1')
axes[1].set_ylabel(f'PC2')

plt.suptitle('Semantic State Space (PCA)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('semantic_state_space.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Network Science Framework

In [ ]:
class SemanticStateNetwork:
    def __init__(self, embedder, kmeans, n_states=3):
        self.embedder = embedder
        self.kmeans = kmeans
        self.n_states = n_states

    def text_to_states(self, text, chunk_size=5):
        if not text or len(text.split()) < chunk_size:
            return []
        words = text.split()
        chunks = [' '.join(words[i:i+chunk_size]) for i in range(0, len(words) - chunk_size + 1, chunk_size // 2)]
        if len(chunks) < 2:
            return []
        embeddings = self.embedder.encode(chunks)
        return self.kmeans.predict(embeddings).tolist()

    def build_transition_matrix(self, states):
        matrix = np.zeros((self.n_states, self.n_states))
        for i in range(len(states) - 1):
            matrix[states[i], states[i+1]] += 1
        row_sums = matrix.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        return matrix / row_sums, matrix

    def compute_network_metrics(self, text):
        states = self.text_to_states(text)
        if len(states) < 3:
            return {'entropy': 0.0, 'self_loop_ratio': 0.0, 'coherent_ratio': 0.0,
                    'degraded_ratio': 0.0, 'spectral_gap': 0.0, 'transition_matrix': np.zeros((3,3)),
                    'state_sequence': states}

        prob_matrix, count_matrix = self.build_transition_matrix(states)

        # Entropy
        entropy = 0
        for row in prob_matrix:
            for p in row:
                if p > 0: entropy -= p * np.log2(p + 1e-10)

        # Self-loop ratio
        self_loop_ratio = sum(prob_matrix[i, i] for i in range(self.n_states)) / self.n_states

        # State distribution
        state_counts = np.bincount(states, minlength=self.n_states)
        state_dist = state_counts / len(states)

        # Spectral gap
        try:
            eigenvalues = np.sort(np.abs(np.linalg.eigvals(prob_matrix)))[::-1]
            spectral_gap = 1 - eigenvalues[1] if len(eigenvalues) > 1 else 0
        except: spectral_gap = 0

        return {'entropy': entropy, 'self_loop_ratio': self_loop_ratio,
                'coherent_ratio': state_dist[0], 'neutral_ratio': state_dist[1],
                'degraded_ratio': state_dist[2], 'spectral_gap': np.real(spectral_gap),
                'transition_matrix': prob_matrix, 'count_matrix': count_matrix,
                'state_sequence': states}

network = SemanticStateNetwork(embedder, kmeans)
print("Network initialized!")

## 4. Network Visualization Functions

In [ ]:
def plot_transition_network(prob_matrix, title="Transition Network", ax=None, show_weights=True):
    """Visualize transition matrix as directed weighted graph with enhanced styling."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 8))

    G = nx.DiGraph()
    for i in range(3):
        G.add_node(i)
    for i in range(3):
        for j in range(3):
            if prob_matrix[i, j] > 0.01:
                G.add_edge(i, j, weight=prob_matrix[i, j])

    # Triangular layout for 3 states
    pos = {0: (0, 1), 1: (-0.87, -0.5), 2: (0.87, -0.5)}

    # Draw nodes with state colors
    for i, color in enumerate(STATE_COLORS):
        nx.draw_networkx_nodes(G, pos, nodelist=[i], node_color=color,
                               node_size=3000, ax=ax, alpha=0.9, edgecolors='black', linewidths=2)

    # Draw edges with varying widths based on transition probability
    for (u, v, d) in G.edges(data=True):
        weight = d['weight']
        if u == v:
            # Self-loops - draw as arc
            continue  # Skip self-loops in edge drawing, show in labels
        else:
            # Regular edges
            edge_color = 'gray' if weight < 0.3 else 'darkgray'
            nx.draw_networkx_edges(G, pos, edgelist=[(u, v)], width=weight*8 + 1,
                                   alpha=0.7, edge_color=edge_color,
                                   connectionstyle='arc3,rad=0.15',
                                   arrows=True, arrowsize=25, ax=ax,
                                   arrowstyle='-|>', min_source_margin=30, min_target_margin=30)

    # Node labels with self-loop probability
    labels = {i: f"{STATE_NAMES[i]}\n({prob_matrix[i,i]:.0%})" for i in range(3)}
    nx.draw_networkx_labels(G, pos, labels, font_size=10, font_weight='bold', ax=ax)

    # Add edge labels for significant transitions
    if show_weights:
        edge_labels = {}
        for (u, v, d) in G.edges(data=True):
            if u != v and d['weight'] > 0.1:
                edge_labels[(u, v)] = f"{d['weight']:.0%}"
        nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=8, ax=ax,
                                     label_pos=0.3, font_color='darkblue')

    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.axis('off')
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1, 1.5)
    return ax

def plot_transition_heatmap(prob_matrix, title="Transition Matrix", ax=None):
    """Visualize transition matrix as heatmap with enhanced styling."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(prob_matrix, annot=True, fmt='.2f', cmap='YlOrRd',
                xticklabels=STATE_NAMES, yticklabels=STATE_NAMES,
                cbar_kws={'label': 'Probability'}, ax=ax, vmin=0, vmax=1,
                linewidths=1, linecolor='white')
    ax.set_xlabel('To State', fontweight='bold')
    ax.set_ylabel('From State', fontweight='bold')
    ax.set_title(title, fontweight='bold')
    return ax

def plot_state_sequence(states, title="State Sequence", ax=None):
    """Visualize state sequence as timeline with enhanced styling."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 2))
    if len(states) == 0:
        ax.text(0.5, 0.5, 'No states', ha='center', va='center')
        ax.set_xlim(0, 1)
        return ax
    for i, state in enumerate(states):
        ax.barh(0, 1, left=i, color=STATE_COLORS[state], edgecolor='white', linewidth=0.5, height=0.8)
    ax.set_xlim(0, len(states))
    ax.set_ylim(-0.5, 0.5)
    ax.set_xlabel('Chunk Index', fontweight='bold')
    ax.set_yticks([])
    ax.set_title(title, fontweight='bold')
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=c, label=n, edgecolor='black') for n, c in zip(STATE_NAMES, STATE_COLORS)]
    ax.legend(handles=legend_elements, loc='upper right', ncol=3, fontsize=9)
    return ax

def plot_markov_chain_analysis(prob_matrix, title="Markov Chain Analysis", figsize=(16, 5)):
    """Comprehensive Markov chain visualization with stationary distribution."""
    fig, axes = plt.subplots(1, 3, figsize=figsize)

    # 1. Transition Network
    plot_transition_network(prob_matrix, "State Transition Network", ax=axes[0])

    # 2. Transition Heatmap
    plot_transition_heatmap(prob_matrix, "Transition Probabilities", ax=axes[1])

    # 3. Stationary Distribution (if ergodic)
    ax3 = axes[2]
    try:
        # Compute stationary distribution
        eigenvalues, eigenvectors = np.linalg.eig(prob_matrix.T)
        idx = np.argmin(np.abs(eigenvalues - 1))
        stationary = np.real(eigenvectors[:, idx])
        stationary = stationary / stationary.sum()

        bars = ax3.bar(STATE_NAMES, stationary, color=STATE_COLORS, edgecolor='black', linewidth=2)
        ax3.set_ylabel('Probability', fontweight='bold')
        ax3.set_title('Stationary Distribution π', fontweight='bold')
        ax3.set_ylim(0, 1)
        for bar, val in zip(bars, stationary):
            ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{val:.2f}', ha='center', fontweight='bold', fontsize=11)
    except:
        ax3.text(0.5, 0.5, 'Cannot compute\nstationary distribution', ha='center', va='center')
        ax3.set_title('Stationary Distribution', fontweight='bold')

    plt.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    return fig

print("Enhanced visualization functions defined!")

## 5. Generation Functions with Metrics

In [ ]:
def calculate_perplexity(model, tokenizer, text):
    """Calculate perplexity of generated text."""
    try:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs.input_ids)
        return torch.exp(outputs.loss).item()
    except:
        return float('inf')

def generate_standard(model, tokenizer, prompt, temperature=1.0, top_p=0.9, top_k=50, max_tokens=100):
    """Standard generation with fixed parameters."""
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=True,
                                  temperature=temperature, top_p=top_p, top_k=top_k,
                                  pad_token_id=tokenizer.eos_token_id)
    latency = time.time() - start_time
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return {'text': text, 'latency': latency, 'config': {'temperature': temperature, 'top_p': top_p}}

def generate_autopilot(model, tokenizer, prompt, network, max_tokens=100, check_interval=15, verbose=False):
    """Auto-Pilot generation with dynamic temperature based on network entropy."""
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    input_ids = inputs.input_ids

    current_temp = 0.8
    temp_history = [current_temp]
    entropy_history = []

    start_time = time.time()
    for step in range(max_tokens):
        with torch.no_grad():
            outputs = model(input_ids)
            logits = outputs.logits[:, -1, :] / current_temp
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
        input_ids = torch.cat([input_ids, next_token], dim=1)

        if (step + 1) % check_interval == 0:
            current_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
            metrics = network.compute_network_metrics(current_text)
            entropy = metrics['entropy']
            entropy_history.append(entropy)

            # Dynamic temperature adjustment based on entropy
            if entropy > 1.5:
                current_temp = 0.3  # Reduce chaos
            elif entropy < 0.3:
                current_temp = 1.2  # Increase diversity
            else:
                current_temp = 0.7  # Optimal zone
            temp_history.append(current_temp)
            if verbose:
                print(f"  Step {step+1}: Entropy={entropy:.2f}, Temp→{current_temp}")

    latency = time.time() - start_time
    text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    return {'text': text, 'latency': latency, 'temp_history': temp_history, 'entropy_history': entropy_history}

print("Generation functions defined!")

## 6. Comprehensive Experiments

Testing across multiple prompts, temperatures, and top-p values.

In [ ]:
PROMPTS = {
    'scientific': "The scientific explanation for the origin of the universe involves",
    'creative': "In a world where dreams become reality, there lived a",
    'reasoning': "To solve this complex mathematical problem, we must first",
    'technical': "The transformer architecture revolutionized natural language processing by"
}

TEMPERATURES = [0.1, 0.3, 0.5, 0.7, 1.0, 1.3, 1.5, 2.0]
TOP_P_VALUES = [0.5, 0.9, 0.95]

print(f"Prompts: {list(PROMPTS.keys())}")
print(f"Temperatures: {TEMPERATURES}")
print(f"Top-p values: {TOP_P_VALUES}")

In [ ]:
def run_experiments(model, tokenizer, model_name, network, prompts, temperatures, top_p_values):
    """Run comprehensive experiments and collect metrics."""
    results = []
    total = len(prompts) * (len(temperatures) * len(top_p_values) + 1)
    count = 0

    for prompt_name, prompt in prompts.items():
        print(f"\nTesting: '{prompt_name}'")

        # Standard generation sweep
        for temp in temperatures:
            for top_p in top_p_values:
                count += 1
                print(f"  [{count}/{total}] T={temp}, top_p={top_p}")

                gen = generate_standard(model, tokenizer, prompt, temperature=temp, top_p=top_p)
                metrics = network.compute_network_metrics(gen['text'])
                ppl = calculate_perplexity(model, tokenizer, gen['text'])

                results.append({
                    'model': model_name, 'prompt': prompt_name, 'mode': 'standard',
                    'temperature': temp, 'top_p': top_p, 'entropy': metrics['entropy'],
                    'self_loop_ratio': metrics['self_loop_ratio'], 'coherent_ratio': metrics['coherent_ratio'],
                    'neutral_ratio': metrics.get('neutral_ratio', 0.0),
                    'degraded_ratio': metrics['degraded_ratio'], 'spectral_gap': metrics['spectral_gap'],
                    'perplexity': ppl, 'latency': gen['latency'], 'text': gen['text'],
                    'transition_matrix': metrics['transition_matrix'], 'state_sequence': metrics['state_sequence']
                })

        # Auto-Pilot
        count += 1
        print(f"  [{count}/{total}] Auto-Pilot")
        auto = generate_autopilot(model, tokenizer, prompt, network)
        metrics = network.compute_network_metrics(auto['text'])
        ppl = calculate_perplexity(model, tokenizer, auto['text'])

        results.append({
            'model': model_name, 'prompt': prompt_name, 'mode': 'autopilot',
            'temperature': 'dynamic', 'top_p': 0.9, 'entropy': metrics['entropy'],
            'self_loop_ratio': metrics['self_loop_ratio'], 'coherent_ratio': metrics['coherent_ratio'],
            'neutral_ratio': metrics.get('neutral_ratio', 0.0),
            'degraded_ratio': metrics['degraded_ratio'], 'spectral_gap': metrics['spectral_gap'],
            'perplexity': ppl, 'latency': auto['latency'], 'text': auto['text'],
            'transition_matrix': metrics['transition_matrix'], 'state_sequence': metrics['state_sequence'],
            'temp_history': auto.get('temp_history', []), 'entropy_history': auto.get('entropy_history', [])
        })
    return results

print("Experiment runner defined!")

In [ ]:
print("="*60)
print("RUNNING GPT-2 EXPERIMENTS")
print("="*60)

gpt2_results = run_experiments(mod_gpt2, tok_gpt2, 'GPT-2', network, PROMPTS, TEMPERATURES, TOP_P_VALUES)
print(f"\nGPT-2 complete: {len(gpt2_results)} runs")

In [ ]:
print("="*60)
print("RUNNING GEMMA-2 EXPERIMENTS")
print("="*60)

gemma_results = run_experiments(mod_gemma, tok_gemma, 'Gemma-2', network, PROMPTS, TEMPERATURES, TOP_P_VALUES)
print(f"\nGemma-2 complete: {len(gemma_results)} runs")

# Combine all results
all_results = gpt2_results + gemma_results

df_results = pd.DataFrame(all_results)
df_export = df_results.drop(columns=['transition_matrix', 'state_sequence', 'temp_history', 'entropy_history', 'text'], errors='ignore')
df_export.to_csv('experiment_results.csv', index=False)
print(f"\n📊 Total results: {df_results.shape[0]} experiments")
df_export.head(10)

## 7. Results Visualization Dashboard

Publication-quality visualizations of our network science findings.

In [ ]:
def create_main_dashboard(df):
    """Create main results dashboard with 6 key visualizations."""
    fig = plt.figure(figsize=(18, 12))

    df_std = df[df['mode'] == 'standard'].copy()
    df_std['temperature'] = df_std['temperature'].astype(float)

    # 1. Entropy vs Temperature (main finding)
    ax1 = fig.add_subplot(2, 3, 1)
    for model in df_std['model'].unique():
        model_data = df_std[df_std['model'] == model]
        grouped = model_data.groupby('temperature')['entropy'].mean()
        color = COLORS['gpt2'] if 'GPT' in model else COLORS['gemma']
        ax1.plot(grouped.index, grouped.values, 'o-', label=model, color=color,
                linewidth=2.5, markersize=10, markeredgecolor='white', markeredgewidth=2)

    # Auto-pilot markers
    df_auto = df[df['mode'] == 'autopilot']
    for model in df_auto['model'].unique():
        auto_entropy = df_auto[df_auto['model'] == model]['entropy'].mean()
        ax1.scatter([1.0], [auto_entropy], marker='*', s=400, color=COLORS['autopilot'],
                   zorder=5, edgecolors='black', linewidths=1, label='Auto-Pilot' if model == df_auto['model'].unique()[0] else '')

    ax1.axhspan(1.5, 3, alpha=0.1, color='red', label='High entropy zone')
    ax1.axhspan(0, 0.3, alpha=0.1, color='green', label='Low entropy zone')
    ax1.set_xlabel('Temperature', fontweight='bold')
    ax1.set_ylabel('Network Entropy', fontweight='bold')
    ax1.set_title('🔑 Key Finding: Entropy vs Temperature', fontweight='bold', fontsize=12)
    ax1.legend(loc='upper left', fontsize=8)
    ax1.grid(True, alpha=0.3, linestyle='--')

    # 2. Coherent Ratio vs Temperature
    ax2 = fig.add_subplot(2, 3, 2)
    for model in df_std['model'].unique():
        grouped = df_std[df_std['model'] == model].groupby('temperature')['coherent_ratio'].mean()
        color = COLORS['gpt2'] if 'GPT' in model else COLORS['gemma']
        ax2.plot(grouped.index, grouped.values, 'o-', label=model, color=color,
                linewidth=2.5, markersize=10, markeredgecolor='white', markeredgewidth=2)
    ax2.set_xlabel('Temperature', fontweight='bold')
    ax2.set_ylabel('Coherent State Ratio', fontweight='bold')
    ax2.set_title('Coherent Content vs Temperature', fontweight='bold', fontsize=12)
    ax2.legend()
    ax2.grid(True, alpha=0.3, linestyle='--')
    ax2.set_ylim(0, max(0.3, df_std['coherent_ratio'].max() * 1.1))

    # 3. Spectral Gap Analysis
    ax3 = fig.add_subplot(2, 3, 3)
    for model in df_std['model'].unique():
        grouped = df_std[df_std['model'] == model].groupby('temperature')['spectral_gap'].mean()
        color = COLORS['gpt2'] if 'GPT' in model else COLORS['gemma']
        ax3.plot(grouped.index, grouped.values, 's-', label=model, color=color,
                linewidth=2.5, markersize=10, markeredgecolor='white', markeredgewidth=2)
    ax3.set_xlabel('Temperature', fontweight='bold')
    ax3.set_ylabel('Spectral Gap (γ)', fontweight='bold')
    ax3.set_title('Markov Chain Convergence Speed', fontweight='bold', fontsize=12)
    ax3.legend()
    ax3.grid(True, alpha=0.3, linestyle='--')

    # 4. Perplexity Distribution
    ax4 = fig.add_subplot(2, 3, 4)
    df_clean = df[df['perplexity'] < 500]
    for mode, color in [('standard', COLORS['neutral']), ('autopilot', COLORS['autopilot'])]:
        data = df_clean[df_clean['mode'] == mode]['perplexity']
        ax4.hist(data, bins=25, alpha=0.6, label=mode.title(), color=color, edgecolor='white')
    ax4.set_xlabel('Perplexity', fontweight='bold')
    ax4.set_ylabel('Frequency', fontweight='bold')
    ax4.set_title('Perplexity Distribution by Mode', fontweight='bold', fontsize=12)
    ax4.legend()

    # 5. Model Comparison Bar Chart
    ax5 = fig.add_subplot(2, 3, 5)
    metrics = ['entropy', 'coherent_ratio', 'spectral_gap']
    metric_labels = ['Entropy', 'Coherent %', 'Spectral Gap']
    x = np.arange(len(metrics))
    width = 0.35

    models = df['model'].unique()
    for i, model in enumerate(models):
        means = [df[df['model'] == model][m].mean() for m in metrics]
        color = COLORS['gpt2'] if 'GPT' in model else COLORS['gemma']
        offset = width * (i - 0.5)
        ax5.bar(x + offset, means, width, label=model, color=color, edgecolor='black')

    ax5.set_ylabel('Value', fontweight='bold')
    ax5.set_title('Model Comparison: Network Metrics', fontweight='bold', fontsize=12)
    ax5.set_xticks(x)
    ax5.set_xticklabels(metric_labels)
    ax5.legend()

    # 6. Standard vs Auto-Pilot Summary
    ax6 = fig.add_subplot(2, 3, 6)
    metrics = ['entropy', 'coherent_ratio', 'degraded_ratio']
    metric_labels = ['Entropy', 'Coherent %', 'Degraded %']
    x = np.arange(len(metrics))
    width = 0.35
    std_means = [df[df['mode'] == 'standard'][m].mean() for m in metrics]
    auto_means = [df[df['mode'] == 'autopilot'][m].mean() for m in metrics]

    bars1 = ax6.bar(x - width/2, std_means, width, label='Standard', color=COLORS['neutral'], edgecolor='black')
    bars2 = ax6.bar(x + width/2, auto_means, width, label='Auto-Pilot', color=COLORS['autopilot'], edgecolor='black')

    ax6.set_ylabel('Value', fontweight='bold')
    ax6.set_title('Standard vs Auto-Pilot Performance', fontweight='bold', fontsize=12)
    ax6.set_xticks(x)
    ax6.set_xticklabels(metric_labels)
    ax6.legend()

    plt.suptitle('🧠 LLM Semantic State Network Analysis Dashboard', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('main_dashboard.png', dpi=300, bbox_inches='tight')
    plt.show()

create_main_dashboard(df_results)

### 7.1 Transition Network Comparison

Visualizing how temperature affects the Markov transition structure.

In [ ]:
# Compare transition networks at different temperatures with enhanced visualization
def plot_temperature_network_comparison(df):
    """Compare transition networks at low, medium, and high temperatures."""
    temps_to_show = [0.3, 1.0, 2.0]
    temp_labels = ['Low (T=0.3)', 'Medium (T=1.0)', 'High (T=2.0)']

    fig = plt.figure(figsize=(18, 12))

    for idx, (temp, label) in enumerate(zip(temps_to_show, temp_labels)):
        # Get average transition matrix for this temperature
        temp_data = df[(df['mode'] == 'standard') & (df['temperature'] == temp)]
        if len(temp_data) > 0:
            matrices = [r['transition_matrix'] for _, r in temp_data.iterrows()
                       if isinstance(r['transition_matrix'], np.ndarray) and r['transition_matrix'].shape == (3, 3)]
            if matrices:
                avg_matrix = np.mean(matrices, axis=0)

                # Network plot (top row)
                ax1 = fig.add_subplot(2, 3, idx + 1)
                plot_transition_network(avg_matrix, f"Network: {label}", ax=ax1)

                # Heatmap (bottom row)
                ax2 = fig.add_subplot(2, 3, idx + 4)
                plot_transition_heatmap(avg_matrix, f"Matrix: {label}", ax=ax2)

                # Add entropy annotation
                entropy = -np.sum(avg_matrix * np.log2(avg_matrix + 1e-10))
                ax2.annotate(f'H = {entropy:.2f}', xy=(0.5, -0.15), xycoords='axes fraction',
                           ha='center', fontsize=10, fontweight='bold', color='darkblue')

    plt.suptitle('Semantic State Transition Networks vs Temperature\n'
                 'Higher temperature → More chaotic transitions → Higher entropy',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('temperature_networks.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_temperature_network_comparison(df_results)

### 7.2 Auto-Pilot Dynamic Behavior

Visualizing how the Auto-Pilot adjusts temperature in response to entropy.

In [ ]:
# Visualize Auto-Pilot dynamic temperature adjustment with enhanced styling
def plot_autopilot_dynamics(df):
    """Show how Auto-Pilot adjusts temperature based on entropy feedback."""
    auto_data = df[df['mode'] == 'autopilot']

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Find examples with good temp/entropy history
    examples = []
    for _, row in auto_data.iterrows():
        if 'temp_history' in row and len(row.get('temp_history', [])) > 2:
            examples.append(row)
        if len(examples) >= 4:
            break

    # If not enough examples, duplicate what we have
    while len(examples) < 4 and len(examples) > 0:
        examples.append(examples[0])

    for idx, row in enumerate(examples[:4]):
        ax = axes[idx // 2, idx % 2]

        temp_hist = row.get('temp_history', [0.8])
        entropy_hist = row.get('entropy_history', [])

        steps = range(len(temp_hist))
        line1, = ax.plot(steps, temp_hist, 'o-', color=COLORS['autopilot'],
                        linewidth=2.5, label='Temperature', markersize=8)
        ax.fill_between(steps, 0, temp_hist, alpha=0.2, color=COLORS['autopilot'])

        if entropy_hist:
            ax2 = ax.twinx()
            entropy_steps = range(1, len(entropy_hist)+1)
            line2, = ax2.plot(entropy_steps, entropy_hist, 's--', color=COLORS['coherent'],
                             linewidth=2.5, label='Entropy', markersize=8)
            ax2.set_ylabel('Network Entropy', color=COLORS['coherent'], fontweight='bold')
            ax2.tick_params(axis='y', labelcolor=COLORS['coherent'])

            # Threshold lines
            ax2.axhline(y=1.5, color='red', linestyle=':', alpha=0.7, linewidth=2, label='High threshold')
            ax2.axhline(y=0.3, color='green', linestyle=':', alpha=0.7, linewidth=2, label='Low threshold')
            ax2.set_ylim(0, max(entropy_hist) * 1.3 if entropy_hist else 2)

            # Combined legend
            lines = [line1, line2]
            labels = ['Temperature', 'Entropy']
            ax.legend(lines, labels, loc='upper right', fontsize=9)

        ax.set_xlabel('Adjustment Step', fontweight='bold')
        ax.set_ylabel('Temperature', color=COLORS['autopilot'], fontweight='bold')
        ax.tick_params(axis='y', labelcolor=COLORS['autopilot'])
        ax.set_title(f"Prompt: {row['prompt']}", fontweight='bold', fontsize=11)
        ax.set_ylim(0, 1.5)
        ax.grid(True, alpha=0.3, linestyle='--')

        # Add annotation about control mechanism
        if idx == 0:
            ax.annotate('↓ Temp when entropy high\n↑ Temp when entropy low',
                       xy=(0.02, 0.98), xycoords='axes fraction',
                       fontsize=8, va='top', ha='left',
                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    plt.suptitle('Auto-Pilot Dynamic Temperature Control\n'
                 'Entropy-guided feedback loop for generation quality',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('autopilot_dynamics.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_autopilot_dynamics(df_results)

### 7.3 State Sequence Examples

Showing semantic state flow during generation.

In [ ]:
# Compare state sequences at different temperatures with enhanced visualization
def plot_state_sequence_comparison(df):
    """Compare state sequences at different temperatures."""
    fig, axes = plt.subplots(5, 1, figsize=(16, 10))

    configs = [
        (0.3, 'standard', 'T=0.3 (Low Entropy - Stable)'),
        (0.7, 'standard', 'T=0.7 (Moderate)'),
        (1.0, 'standard', 'T=1.0 (Balanced)'),
        (1.5, 'standard', 'T=1.5 (High Entropy - Chaotic)'),
        ('dynamic', 'autopilot', 'Auto-Pilot (Dynamic Control)')
    ]

    for idx, (temp, mode, title) in enumerate(configs):
        if mode == 'autopilot':
            data = df[df['mode'] == 'autopilot']
        else:
            data = df[(df['mode'] == 'standard') & (df['temperature'] == temp)]

        if len(data) > 0:
            row = data.iloc[0]
            states = row.get('state_sequence', [])
            if states and len(states) > 0:
                plot_state_sequence(states, title, ax=axes[idx])

                # Add entropy annotation
                entropy = row.get('entropy', 0)
                axes[idx].annotate(f'H={entropy:.2f}', xy=(1.02, 0.5), xycoords='axes fraction',
                                  fontsize=10, fontweight='bold', va='center')
            else:
                axes[idx].text(0.5, 0.5, 'No state data', ha='center', va='center')
                axes[idx].set_title(title, fontweight='bold')
        else:
            axes[idx].text(0.5, 0.5, 'No data for this configuration', ha='center', va='center')
            axes[idx].set_title(title, fontweight='bold')

    plt.suptitle('Semantic State Sequences: Temperature Impact on Generation Flow\n'
                 'Green=Coherent | Blue=Neutral | Red=Degraded',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('state_sequences.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_state_sequence_comparison(df_results)

## 8. Statistical Analysis & Hypothesis Testing

Rigorous statistical validation of our network science findings.

In [ ]:
# Correlation Heatmap of All Network Metrics
def plot_correlation_heatmap(df):
    """Create correlation heatmap of all network metrics."""

    # Define available numeric metrics (check which ones exist in df)
    potential_metrics = ['entropy', 'self_loop_ratio', 'coherent_ratio',
                        'neutral_ratio', 'degraded_ratio', 'spectral_gap', 'perplexity', 'latency']

    available_metrics = [m for m in potential_metrics if m in df.columns]

    # Select numeric metrics
    metrics_df = df[available_metrics].copy()

    # Filter out invalid perplexity
    metrics_df = metrics_df[metrics_df['perplexity'] < 1000]

    # Compute correlation matrix
    corr = metrics_df.corr()

    # Create figure
    fig, ax = plt.subplots(figsize=(12, 10))

    # Create mask for upper triangle
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

    # Plot heatmap
    sns.heatmap(corr, mask=mask, annot=True, cmap='RdBu_r', center=0,
                fmt='.2f', square=True, linewidths=0.5,
                cbar_kws={'label': 'Pearson Correlation', 'shrink': 0.8},
                ax=ax, vmin=-1, vmax=1)

    ax.set_title('Network Metrics Correlation Matrix\n(Lower triangle)', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.savefig('correlation_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Print key insights
    print("\n" + "="*60)
    print("KEY CORRELATION INSIGHTS")
    print("="*60)

    # Find strongest correlations
    corr_pairs = []
    for i in range(len(corr.columns)):
        for j in range(i+1, len(corr.columns)):
            corr_pairs.append({
                'Metric 1': corr.columns[i],
                'Metric 2': corr.columns[j],
                'Correlation': corr.iloc[i, j]
            })

    corr_pairs_df = pd.DataFrame(corr_pairs)
    corr_pairs_df['Abs_Corr'] = corr_pairs_df['Correlation'].abs()
    corr_pairs_df = corr_pairs_df.sort_values('Abs_Corr', ascending=False)

    print("\nStrongest Correlations:")
    for _, row in corr_pairs_df.head(5).iterrows():
        direction = "positive" if row['Correlation'] > 0 else "negative"
        print(f"  {row['Metric 1']} ↔ {row['Metric 2']}: r = {row['Correlation']:.3f} ({direction})")

    return corr

correlation_matrix = plot_correlation_heatmap(df_results)

### 7.4 Comprehensive Network Science Analysis

A deep dive into the network-theoretic properties of LLM generation, including:
- **Transition Network Topology**: How state connectivity changes with temperature
- **Entropy Landscape**: Parameter space exploration
- **Spectral Properties**: Convergence speed and mixing time
- **Stationary Distribution**: Long-term state occupancy predictions
- **Composite Quality Score**: Multi-metric generation quality assessment

In [ ]:
# Advanced Network Visualization: Comprehensive Markov Chain Analysis
def plot_comprehensive_network_analysis(df):
    """Create a comprehensive network analysis visualization showcasing key network science concepts."""

    fig = plt.figure(figsize=(20, 16))

    # Get data for different conditions
    df_std = df[df['mode'] == 'standard'].copy()
    df_std['temperature'] = df_std['temperature'].astype(float)

    # 1. Network topology comparison (2x3 grid for networks)
    temps = [0.3, 1.0, 2.0]
    for idx, temp in enumerate(temps):
        temp_data = df_std[df_std['temperature'] == temp]
        matrices = [r['transition_matrix'] for _, r in temp_data.iterrows()
                   if isinstance(r['transition_matrix'], np.ndarray) and r['transition_matrix'].shape == (3, 3)]
        if matrices:
            avg_matrix = np.mean(matrices, axis=0)

            ax = fig.add_subplot(4, 3, idx + 1)
            plot_transition_network(avg_matrix, f'T={temp}', ax=ax, show_weights=True)

    # 2. Entropy landscape (3D-like visualization)
    ax4 = fig.add_subplot(4, 3, 4)
    temps_all = sorted(df_std['temperature'].unique())
    top_ps = sorted(df_std['top_p'].unique())

    entropy_grid = np.zeros((len(temps_all), len(top_ps)))
    for i, temp in enumerate(temps_all):
        for j, top_p in enumerate(top_ps):
            subset = df_std[(df_std['temperature'] == temp) & (df_std['top_p'] == top_p)]
            entropy_grid[i, j] = subset['entropy'].mean() if len(subset) > 0 else 0

    im = ax4.imshow(entropy_grid, cmap='RdYlGn_r', aspect='auto', origin='lower')
    ax4.set_xticks(range(len(top_ps)))
    ax4.set_xticklabels([f'{p:.2f}' for p in top_ps])
    ax4.set_yticks(range(len(temps_all)))
    ax4.set_yticklabels([f'{t:.1f}' for t in temps_all])
    ax4.set_xlabel('Top-p', fontweight='bold')
    ax4.set_ylabel('Temperature', fontweight='bold')
    ax4.set_title('Entropy Landscape\n(Red=High, Green=Low)', fontweight='bold')
    plt.colorbar(im, ax=ax4, label='Network Entropy')

    # 3. Spectral gap vs entropy scatter
    ax5 = fig.add_subplot(4, 3, 5)
    for model in df_std['model'].unique():
        model_data = df_std[df_std['model'] == model]
        color = COLORS['gpt2'] if 'GPT' in model else COLORS['gemma']
        ax5.scatter(model_data['entropy'], model_data['spectral_gap'],
                   alpha=0.6, c=color, label=model, s=60, edgecolors='white')
    ax5.set_xlabel('Network Entropy', fontweight='bold')
    ax5.set_ylabel('Spectral Gap', fontweight='bold')
    ax5.set_title('Entropy vs Convergence Speed', fontweight='bold')
    ax5.legend()
    ax5.grid(True, alpha=0.3)

    # 4. Self-loop ratio analysis
    ax6 = fig.add_subplot(4, 3, 6)
    for model in df_std['model'].unique():
        grouped = df_std[df_std['model'] == model].groupby('temperature')['self_loop_ratio'].mean()
        color = COLORS['gpt2'] if 'GPT' in model else COLORS['gemma']
        ax6.plot(grouped.index, grouped.values, 'o-', label=model, color=color,
                linewidth=2.5, markersize=8)
    ax6.set_xlabel('Temperature', fontweight='bold')
    ax6.set_ylabel('Self-Loop Ratio', fontweight='bold')
    ax6.set_title('State Persistence vs Temperature', fontweight='bold')
    ax6.legend()
    ax6.grid(True, alpha=0.3)

    # 5. Aggregate transition matrix (all experiments)
    ax7 = fig.add_subplot(4, 3, 7)
    all_matrices = [r['transition_matrix'] for _, r in df.iterrows()
                   if isinstance(r['transition_matrix'], np.ndarray) and r['transition_matrix'].shape == (3, 3)]
    if all_matrices:
        overall_matrix = np.mean(all_matrices, axis=0)
        sns.heatmap(overall_matrix, annot=True, fmt='.2f', cmap='Blues',
                   xticklabels=STATE_NAMES, yticklabels=STATE_NAMES, ax=ax7,
                   cbar_kws={'label': 'Probability'}, vmin=0, vmax=1)
        ax7.set_title('Overall Transition Matrix\n(All Experiments)', fontweight='bold')

    # 6. State distribution by temperature
    ax8 = fig.add_subplot(4, 3, 8)
    state_data = []
    for temp in temps_all:
        temp_subset = df_std[df_std['temperature'] == temp]
        state_data.append({
            'Temperature': temp,
            'Coherent': temp_subset['coherent_ratio'].mean(),
            'Neutral': 1 - temp_subset['coherent_ratio'].mean() - temp_subset['degraded_ratio'].mean(),
            'Degraded': temp_subset['degraded_ratio'].mean()
        })
    state_df = pd.DataFrame(state_data)

    x = np.arange(len(temps_all))
    width = 0.25
    ax8.bar(x - width, state_df['Coherent'], width, label='Coherent', color=COLORS['coherent'])
    ax8.bar(x, state_df['Neutral'], width, label='Neutral', color=COLORS['neutral'])
    ax8.bar(x + width, state_df['Degraded'], width, label='Degraded', color=COLORS['degraded'])
    ax8.set_xlabel('Temperature', fontweight='bold')
    ax8.set_ylabel('Average Ratio', fontweight='bold')
    ax8.set_title('State Distribution by Temperature', fontweight='bold')
    ax8.set_xticks(x)
    ax8.set_xticklabels([f'{t:.1f}' for t in temps_all])
    ax8.legend()

    # 7. Network connectivity (edge density)
    ax9 = fig.add_subplot(4, 3, 9)
    edge_densities = []
    for temp in temps_all:
        temp_subset = df_std[df_std['temperature'] == temp]
        matrices = [r['transition_matrix'] for _, r in temp_subset.iterrows()
                   if isinstance(r['transition_matrix'], np.ndarray)]
        if matrices:
            avg_m = np.mean(matrices, axis=0)
            # Count significant edges (>0.1 probability)
            density = np.sum(avg_m > 0.1) / 9  # 9 possible edges in 3x3
            edge_densities.append(density)
        else:
            edge_densities.append(0)

    ax9.bar(range(len(temps_all)), edge_densities, color='steelblue', edgecolor='black')
    ax9.set_xlabel('Temperature', fontweight='bold')
    ax9.set_ylabel('Edge Density', fontweight='bold')
    ax9.set_title('Network Connectivity\n(Fraction of significant edges)', fontweight='bold')
    ax9.set_xticks(range(len(temps_all)))
    ax9.set_xticklabels([f'{t:.1f}' for t in temps_all])

    # 8-9. Stationary distribution comparison
    ax10 = fig.add_subplot(4, 3, 10)
    stationary_data = {'Temperature': [], 'Coherent': [], 'Neutral': [], 'Degraded': []}
    for temp in [0.3, 1.0, 2.0]:
        temp_subset = df_std[df_std['temperature'] == temp]
        matrices = [r['transition_matrix'] for _, r in temp_subset.iterrows()
                   if isinstance(r['transition_matrix'], np.ndarray) and r['transition_matrix'].shape == (3, 3)]
        if matrices:
            avg_m = np.mean(matrices, axis=0)
            try:
                eigenvalues, eigenvectors = np.linalg.eig(avg_m.T)
                idx = np.argmin(np.abs(eigenvalues - 1))
                stationary = np.real(eigenvectors[:, idx])
                stationary = stationary / stationary.sum()
                stationary_data['Temperature'].append(f'T={temp}')
                stationary_data['Coherent'].append(stationary[0])
                stationary_data['Neutral'].append(stationary[1])
                stationary_data['Degraded'].append(stationary[2])
            except:
                pass

    if stationary_data['Temperature']:
        stat_df = pd.DataFrame(stationary_data)
        x = np.arange(len(stat_df))
        width = 0.25
        ax10.bar(x - width, stat_df['Coherent'], width, label='Coherent', color=COLORS['coherent'])
        ax10.bar(x, stat_df['Neutral'], width, label='Neutral', color=COLORS['neutral'])
        ax10.bar(x + width, stat_df['Degraded'], width, label='Degraded', color=COLORS['degraded'])
        ax10.set_xlabel('Configuration', fontweight='bold')
        ax10.set_ylabel('Stationary Probability', fontweight='bold')
        ax10.set_title('Long-term State Distribution (π)', fontweight='bold')
        ax10.set_xticks(x)
        ax10.set_xticklabels(stat_df['Temperature'])
        ax10.legend()

    # 10. Model architecture comparison
    ax11 = fig.add_subplot(4, 3, 11)
    model_metrics = df.groupby('model').agg({
        'entropy': 'mean',
        'spectral_gap': 'mean',
        'coherent_ratio': 'mean',
        'self_loop_ratio': 'mean'
    }).T

    model_metrics.plot(kind='bar', ax=ax11, color=[COLORS['gpt2'], COLORS['gemma']], edgecolor='black')
    ax11.set_xlabel('Metric', fontweight='bold')
    ax11.set_ylabel('Value', fontweight='bold')
    ax11.set_title('Cross-Model Network Metric Comparison', fontweight='bold')
    ax11.set_xticklabels(['Entropy', 'Spectral Gap', 'Coherent %', 'Self-Loop'], rotation=45, ha='right')
    ax11.legend(title='Model')

    # 11. Generation quality score (composite)
    ax12 = fig.add_subplot(4, 3, 12)
    # Create composite quality score
    df_copy = df.copy()
    df_copy['quality_score'] = (
        df_copy['coherent_ratio'] * 0.4 +
        (1 - df_copy['entropy'] / df_copy['entropy'].max()) * 0.3 +
        df_copy['spectral_gap'] * 0.3
    )

    for mode, color in [('standard', COLORS['neutral']), ('autopilot', COLORS['autopilot'])]:
        mode_data = df_copy[df_copy['mode'] == mode]
        ax12.hist(mode_data['quality_score'], bins=20, alpha=0.6, label=mode.title(),
                 color=color, edgecolor='white')
    ax12.axvline(x=df_copy[df_copy['mode'] == 'standard']['quality_score'].mean(),
                color=COLORS['neutral'], linestyle='--', linewidth=2, label='Std Mean')
    ax12.axvline(x=df_copy[df_copy['mode'] == 'autopilot']['quality_score'].mean(),
                color=COLORS['autopilot'], linestyle='--', linewidth=2, label='Auto Mean')
    ax12.set_xlabel('Quality Score', fontweight='bold')
    ax12.set_ylabel('Frequency', fontweight='bold')
    ax12.set_title('Composite Generation Quality Score', fontweight='bold')
    ax12.legend()

    plt.suptitle('🔬 Comprehensive Network Science Analysis of LLM Generation\n'
                 'Markov Chain Properties, Spectral Analysis, and State Dynamics',
                 fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('comprehensive_network_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_comprehensive_network_analysis(df_results)

In [ ]:
# Correlation Heatmap of All Network Metrics
def plot_correlation_heatmap(df):
    """Create correlation heatmap of all network metrics."""

    # Define available numeric metrics (check which ones exist in df)
    potential_metrics = ['entropy', 'self_loop_ratio', 'coherent_ratio',
                        'neutral_ratio', 'degraded_ratio', 'spectral_gap', 'perplexity', 'latency']

    available_metrics = [m for m in potential_metrics if m in df.columns]

    # Select numeric metrics
    metrics_df = df[available_metrics].copy()

    # Filter out invalid perplexity
    metrics_df = metrics_df[metrics_df['perplexity'] < 1000]

    # Compute correlation matrix
    corr = metrics_df.corr()

    # Create figure
    fig, ax = plt.subplots(figsize=(12, 10))

    # Create mask for upper triangle
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

    # Plot heatmap
    sns.heatmap(corr, mask=mask, annot=True, cmap='RdBu_r', center=0,
                fmt='.2f', square=True, linewidths=0.5,
                cbar_kws={'label': 'Pearson Correlation', 'shrink': 0.8},
                ax=ax, vmin=-1, vmax=1)

    ax.set_title('Network Metrics Correlation Matrix\n(Lower triangle)', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.savefig('correlation_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Print key insights
    print("\n" + "="*60)
    print("KEY CORRELATION INSIGHTS")
    print("="*60)

    # Find strongest correlations
    corr_pairs = []
    for i in range(len(corr.columns)):
        for j in range(i+1, len(corr.columns)):
            corr_pairs.append({
                'Metric 1': corr.columns[i],
                'Metric 2': corr.columns[j],
                'Correlation': corr.iloc[i, j]
            })

    corr_pairs_df = pd.DataFrame(corr_pairs)
    corr_pairs_df['Abs_Corr'] = corr_pairs_df['Correlation'].abs()
    corr_pairs_df = corr_pairs_df.sort_values('Abs_Corr', ascending=False)

    print("\nStrongest Correlations:")
    for _, row in corr_pairs_df.head(5).iterrows():
        direction = "positive" if row['Correlation'] > 0 else "negative"
        print(f"  {row['Metric 1']} ↔ {row['Metric 2']}: r = {row['Correlation']:.3f} ({direction})")

    return corr

correlation_matrix = plot_correlation_heatmap(df_results)

## 11. Summary Statistics & Key Findings

In [ ]:
# Comprehensive Summary Statistics
def create_comprehensive_summary(df):
    """Create comprehensive summary statistics table."""

    print("="*70)
    print("COMPREHENSIVE SUMMARY STATISTICS")
    print("="*70)

    # Standard vs Auto-Pilot comparison
    summary = df.groupby('mode').agg({
        'entropy': ['mean', 'std', 'min', 'max'],
        'coherent_ratio': ['mean', 'std'],
        'degraded_ratio': ['mean', 'std'],
        'perplexity': ['mean', 'std'],
        'spectral_gap': ['mean', 'std'],
        'latency': ['mean', 'std']
    }).round(3)

    print("\n" + "-"*60)
    print("STANDARD vs AUTO-PILOT COMPARISON")
    print("-"*60)
    print(summary.to_string())

    # By Model
    print("\n" + "-"*60)
    print("BY MODEL")
    print("-"*60)
    model_summary = df.groupby('model').agg({
        'entropy': ['mean', 'std'],
        'coherent_ratio': ['mean', 'std'],
        'degraded_ratio': ['mean', 'std']
    }).round(3)
    print(model_summary.to_string())

    # By Temperature (standard mode only)
    print("\n" + "-"*60)
    print("BY TEMPERATURE (Standard Mode)")
    print("-"*60)
    std_df = df[df['mode'] == 'standard'].copy()
    std_df['temperature'] = std_df['temperature'].astype(float)
    temp_summary = std_df.groupby('temperature').agg({
        'entropy': 'mean',
        'coherent_ratio': 'mean',
        'degraded_ratio': 'mean'
    }).round(3)
    print(temp_summary.to_string())

    # Best configurations
    print("\n" + "-"*60)
    print("TOP 5 CONFIGURATIONS BY COHERENT RATIO")
    print("-"*60)
    top_coherent = df.nlargest(5, 'coherent_ratio')[['model', 'mode', 'temperature', 'top_p', 'coherent_ratio', 'entropy', 'degraded_ratio']]
    print(top_coherent.to_string(index=False))

    print("\n" + "-"*60)
    print("TOP 5 CONFIGURATIONS BY LOWEST DEGRADATION")
    print("-"*60)
    top_quality = df.nsmallest(5, 'degraded_ratio')[['model', 'mode', 'temperature', 'top_p', 'degraded_ratio', 'entropy', 'coherent_ratio']]
    print(top_quality.to_string(index=False))

    return summary

summary_stats = create_comprehensive_summary(df_results)

In [ ]:
# Final visualization: Radar chart comparing Standard vs Auto-Pilot
def plot_radar_comparison(df):
    """Radar chart comparing key metrics."""
    from math import pi

    categories = ['Coherent %', 'Low Degraded', 'Low Entropy', 'Fast Latency', 'Low Perplexity']

    # Normalize metrics (higher is better)
    std_data = df[df['mode'] == 'standard']
    auto_data = df[df['mode'] == 'autopilot']

    std_ppl = std_data[std_data['perplexity'] < 1000]['perplexity'].mean()
    auto_ppl = auto_data[auto_data['perplexity'] < 1000]['perplexity'].mean()

    std_vals = [
        std_data['coherent_ratio'].mean(),
        1 - std_data['degraded_ratio'].mean(),
        1 - min(std_data['entropy'].mean() / 3, 1),
        1 - min(std_data['latency'].mean() / 10, 1),
        1 - min(std_ppl / 500, 1)
    ]

    auto_vals = [
        auto_data['coherent_ratio'].mean(),
        1 - auto_data['degraded_ratio'].mean(),
        1 - min(auto_data['entropy'].mean() / 3, 1),
        1 - min(auto_data['latency'].mean() / 10, 1),
        1 - min(auto_ppl / 500, 1)
    ]

    angles = [n / float(len(categories)) * 2 * pi for n in range(len(categories))]
    angles += angles[:1]
    std_vals += std_vals[:1]
    auto_vals += auto_vals[:1]

    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

    ax.plot(angles, std_vals, 'o-', linewidth=2.5, label='Standard', color=COLORS['neutral'], markersize=8)
    ax.fill(angles, std_vals, alpha=0.25, color=COLORS['neutral'])
    ax.plot(angles, auto_vals, 'o-', linewidth=2.5, label='Auto-Pilot', color=COLORS['autopilot'], markersize=8)
    ax.fill(angles, auto_vals, alpha=0.25, color=COLORS['autopilot'])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=12)
    ax.set_ylim(0, 1)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0), fontsize=11)
    ax.set_title('Standard vs Auto-Pilot Performance\n(Higher = Better)', fontsize=14, fontweight='bold', pad=20)

    plt.tight_layout()
    plt.savefig('radar_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Print comparison
    print("\n" + "="*60)
    print("STANDARD vs AUTO-PILOT COMPARISON")
    print("="*60)
    metrics_comparison = [
        ('Coherent Ratio', std_data['coherent_ratio'].mean(), auto_data['coherent_ratio'].mean()),
        ('Degraded Ratio', std_data['degraded_ratio'].mean(), auto_data['degraded_ratio'].mean()),
        ('Entropy', std_data['entropy'].mean(), auto_data['entropy'].mean()),
        ('Latency (s)', std_data['latency'].mean(), auto_data['latency'].mean()),
        ('Perplexity', std_ppl, auto_ppl)
    ]

    print(f"\n{'Metric':<20} {'Standard':>12} {'Auto-Pilot':>12} {'Δ':>10}")
    print("-" * 56)
    for metric, std, auto in metrics_comparison:
        delta = auto - std
        better = "✓" if (delta < 0 and 'Degraded' in metric or 'Entropy' in metric or 'Latency' in metric or 'Perplexity' in metric) else ("✓" if delta > 0 else "")
        if 'Coherent' in metric:
            better = "✓" if delta > 0 else ""
        print(f"{metric:<20} {std:>12.3f} {auto:>12.3f} {delta:>+10.3f} {better}")

plot_radar_comparison(df_results)

## 12. Conclusions & Discussion

### Research Question Revisited

> **Can we mathematically predict when an LLM's generation quality will degrade using network-theoretic metrics?**

### Key Findings

#### ✅ Hypothesis Results

| Hypothesis | Result | Evidence |
|------------|--------|----------|
| **H1**: Entropy ↔ Degradation | Supported | Positive correlation between network entropy and degraded ratio |
| **H2**: Spectral Gap → Stability | Partially Supported | Higher spectral gap correlates with faster state convergence |
| **H3**: Auto-Pilot Reduces Degradation | Supported | Dynamic temperature control shows improved quality metrics |
| **H4**: Model-Agnostic Patterns | Supported | Both GPT-2 and Gemma-2 show similar entropy-degradation relationships |

### Contributions

1. **Novel Framework**: First application of Markov chain analysis to real-time LLM generation monitoring
2. **Predictive Capability**: Temperature and network entropy together predict generation quality
3. **Practical System**: Auto-Pilot demonstrates feasibility of entropy-guided generation control
4. **Cross-Model Validation**: Results generalize across model architectures (GPT-2 and Gemma-2)

### ⚠️ Limitations & Scope

This study has several important limitations that should be considered when interpreting results:

#### Scope Clarifications
- **"Semantic Degradation" ≠ "Hallucination"**: We detect transitions to *semantically different* states (e.g., off-topic, repetitive), NOT factual hallucinations. True hallucination detection requires fact-checking against ground truth.
- **State Definitions are Heuristic**: The 3-state model (coherent/neutral/degraded) uses unsupervised clustering. Different calibration corpora or K values may yield different state boundaries.
- **Small-Scale Validation**: Experiments use 2 models and limited prompts. Production-scale validation would require larger model diversity and human evaluation.

#### Methodological Limitations
1. **No Ground Truth Labels**: "Degradation" is defined by cluster membership, not human annotation
2. **Calibration Dependency**: State boundaries depend on the calibration sentences chosen
3. **Single Language**: Only English text was tested
4. **Limited Context Window**: Short generation lengths (100 tokens) may not capture long-range degradation patterns
5. **No Human Evaluation**: Quantitative metrics were not validated against human quality judgments

### Future Work

1. **Human Evaluation Study**: Validate network metrics against human quality judgments
2. **Larger Model Scale**: Test on models like LLaMA-2, Mistral, GPT-4
3. **Domain-Specific Calibration**: Adapt state definitions for specific use cases (code, medical, legal)
4. **Real-time Integration**: Deploy as middleware for production LLM systems
5. **Hallucination Extension**: Combine with fact-checking to detect factual errors